In [1]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

ModuleNotFoundError: No module named 'xgboost'

In [1]:
import sys

print(sys.executable)


c:\Users\samat\OneDrive\Documents\oxygen-saturation-prediction\.venv\Scripts\python.exe


In [2]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.2.0


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_file = (
    project_root
    / "data"
    / "processed"
    / "multi_patient_50_split.csv"
)

df = pd.read_csv(data_file)

print("Dataset loaded!")
print("Shape:", df.shape)
print()
print(df["split"].value_counts())

Dataset loaded!
Shape: (12515, 24)

split
train         8789
test          1866
validation    1860
Name: count, dtype: int64


In [4]:
spo2_features = [
    "spo2",
    "spo2_lag_1",
    "spo2_lag_3",
    "spo2_lag_5",
    "spo2_change_1min",
    "spo2_change_3min",
    "spo2_rolling_mean_5",
    "spo2_rolling_std_5"
]

target = "future_spo2_5min"

print("Number of features:", len(spo2_features))

Number of features: 8


In [5]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

train_ready = train_df.dropna(
    subset=[target]
).copy()

common_val_df = val_df.dropna(
    subset=["spo2", target]
).copy()

print("Training rows:", len(train_ready))
print("Validation rows:", len(common_val_df))

Training rows: 8310
Validation rows: 1748


In [6]:
from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [7]:
xgb_spo2_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            XGBRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=4,
                min_child_weight=3,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [8]:
X_train = train_ready[spo2_features]
y_train = train_ready[target]

X_val = common_val_df[spo2_features]
y_val = common_val_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print()

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (8310, 8)
y_train: (8310,)

X_val: (1748, 8)
y_val: (1748,)


In [9]:
xgb_spo2_model.fit(
    X_train,
    y_train
)

print("XGBoost training complete!")

XGBoost training complete!


In [10]:
xgb_spo2_pred = xgb_spo2_model.predict(
    X_val
)

print(
    "Predictions created:",
    len(xgb_spo2_pred)
)

Predictions created: 1748


In [11]:
xgb_mae = mean_absolute_error(
    y_val,
    xgb_spo2_pred
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        xgb_spo2_pred
    )
)

xgb_r2 = r2_score(
    y_val,
    xgb_spo2_pred
)

print("=== XGBOOST - SpO2 HISTORY ===")
print()

print(f"MAE:  {xgb_mae:.6f}")
print(f"RMSE: {xgb_rmse:.6f}")
print(f"R²:   {xgb_r2:.6f}")

=== XGBOOST - SpO2 HISTORY ===

MAE:  0.423909
RMSE: 1.362911
R²:   0.118666


In [12]:
xgb_result = {
    "Model": "XGBoost - SpO2 History",
    "MAE": xgb_mae,
    "RMSE": xgb_rmse,
    "R2": xgb_r2
}

results_file = (
    project_root
    / "data"
    / "processed"
    / "model_results.csv"
)

previous_results = pd.read_csv(results_file)

# Remove old XGBoost result if notebook is rerun
previous_results = previous_results[
    ~previous_results["Model"]
    .str.startswith("XGBoost")
].copy()

xgb_results = pd.DataFrame([xgb_result])

all_results = pd.concat(
    [
        previous_results,
        xgb_results
    ],
    ignore_index=True
)

all_results = (
    all_results
    .drop_duplicates(
        subset=["Model"],
        keep="last"
    )
    .reset_index(drop=True)
)

all_results.to_csv(
    results_file,
    index=False
)

print("Rows:", len(all_results))
print(
    "Unique models:",
    all_results["Model"].nunique()
)

all_results.sort_values("MAE")

Rows: 12
Unique models: 12


,Model,MAE,RMSE,R2
0,Persistence Baseline,0.266876,1.657234,-0.303086
2,Ridge - SpO2 History,0.420457,1.362850,0.118745
1,Linear Regression - SpO2 History,0.420464,1.362954,0.118610
11,XGBoost - SpO2 History,0.423909,1.362911,0.118666
3,Ridge - SpO2 + HR,0.425369,1.363505,0.117898
5,Random Forest - SpO2 History,0.425950,1.415256,0.049668
8,Gradient Boosting - SpO2 History,0.431746,1.361006,0.121128
9,Gradient Boosting - SpO2 + HR,0.441540,1.348987,0.136582
6,Random Forest - SpO2 + HR,0.445252,1.436580,0.020815
7,Random Forest - All Vitals,0.472970,1.435870,0.021782
